In [ ]:
# =========================================
# 1. IMPORTS
# =========================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import r2_score, mean_squared_error, accuracy_score

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# =========================================
# 2. CREATE FOLDERS
# =========================================
os.makedirs("/home/jae/Desktop/derivarable 2/results/plots", exist_ok=True)

# =========================================
# 3. LOAD DATA
# =========================================
df = pd.read_csv("/home/jae/Desktop/derivarable 2/Crop-Yield-Prediction-based-on-Indian-Agriculture/Crop Prediction dataset.csv")
df = df.dropna()

# =========================================
# 4. FEATURES
# =========================================
categorical_cols = ['State_Name','District_Name','Season','Crop']
numerical_cols = ['Temperature','Humidity','Soil_Moisture','Area']

X = df[categorical_cols + numerical_cols]
y = df['Production']

# =========================================
# 5. SPLIT
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================
# 6. PREPROCESSOR
# =========================================
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ('num', StandardScaler(), numerical_cols)
])

# =========================================
# 7. RANDOM FOREST
# =========================================
rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=50, random_state=42))
])

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("\n--- Random Forest ---")
print("R2:", r2_score(y_test, rf_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))

# Plot
plt.figure()
plt.scatter(y_test, rf_pred)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Random Forest Prediction")
plt.savefig("results/plots/random_forest_prediction.png")
plt.close()

# =========================================
# 8. DECISION TREE
# =========================================
dt = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(max_depth=10))
])

dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

print("\n--- Decision Tree ---")
print("R2:", r2_score(y_test, dt_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, dt_pred)))

# Plot
plt.figure()
plt.scatter(y_test, dt_pred)
plt.title("Decision Tree Prediction")
plt.savefig("results/plots/decision_tree_prediction.png")
plt.close()

# =========================================
# 9. SVR
# =========================================
svr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SVR(kernel='rbf'))
])

svr.fit(X_train, y_train)
svr_pred = svr.predict(X_test)

print("\n--- SVR ---")
print("R2:", r2_score(y_test, svr_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, svr_pred)))

# Plot
plt.figure()
plt.scatter(y_test, svr_pred)
plt.title("SVR Prediction")
plt.savefig("results/plots/svr_prediction.png")
plt.close()

# =========================================
# 10. NAIVE BAYES
# =========================================
y_cat = pd.qcut(y, q=3, labels=[0,1,2])

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cat, test_size=0.2, random_state=42
)

nb = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ])),
    ('model', GaussianNB())
])

nb.fit(X_train_c, y_train_c)
nb_pred = nb.predict(X_test_c)

print("\n--- Naive Bayes ---")
print("Accuracy:", accuracy_score(y_test_c, nb_pred))

# =========================================
# 11. CORRELATION HEATMAP
# =========================================
numeric_data = df.select_dtypes(include=['number'])

plt.figure(figsize=(10,6))
sns.heatmap(numeric_data.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title("Correlation Matrix")
plt.savefig("results/plots/correlation_matrix.png")
plt.close()

# =========================================
# 12. CLUSTERING
# =========================================
X_cluster = df[numerical_cols]
X_scaled = StandardScaler().fit_transform(X_cluster)

# KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X_scaled)

plt.figure()
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=labels)
plt.title("KMeans Clustering")
plt.savefig("results/plots/kmeans.png")
plt.close()

# DBSCAN
db = DBSCAN(eps=1.5, min_samples=5)
db_labels = db.fit_predict(X_scaled)

plt.figure()
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=db_labels)
plt.title("DBSCAN Clustering")
plt.savefig("results/plots/dbscan.png")
plt.close()

# Hierarchical
agg = AgglomerativeClustering(n_clusters=3)
agg_labels = agg.fit_predict(X_scaled)

plt.figure()
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=agg_labels)
plt.title("Hierarchical Clustering")
plt.savefig("results/plots/hierarchical.png")
plt.close()

# =========================================
# 13. OUTLIER DETECTION
# =========================================
iso = IsolationForest(contamination=0.05)
outliers = iso.fit_predict(X_scaled)

plt.figure()
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=outliers)
plt.title("Isolation Forest Outliers")
plt.savefig("results/plots/outliers.png")
plt.close()

print("\nAll plots saved in: results/plots/")


--- Random Forest ---
R2: 0.80940715916973
RMSE: 3301430.877354847

--- Decision Tree ---
R2: 0.7561560829553067
RMSE: 3734262.8160572173

--- SVR ---
R2: -0.0014245392248386501
RMSE: 7567597.616784303

--- Naive Bayes ---
Accuracy: 0.6148438284623883
